In [1]:
import hoda
import tensorly as tl

tl.set_backend('cupy', local_threadsafe=False)
print(tl.get_backend())
%pip install toeplitzlda
%pip freeze | grep moabb

cupy

[notice] A new release of pip is available: 23.2.1 -> 24.0
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
moabb==1.0.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

paradigm = P300(resample=48)
datasets = [
    #BI2012(),
    #BI2013a(),
    #BI2014a(),
    #BI2014b(),
    #BI2015a(),
    #BI2015b(),
    BNCI2014_008(),
    #BNCI2014_009(),
    #EPFLP300(),
    #BNCI2015_003(),
    #Cattan2019_VR(),
    #Huebner2017(),
    #Huebner2018(),
    #Lee2019_ERP(),
    #Sosulski2019(),
]

evaluation = WithinSessionEvaluation(
    paradigm=paradigm,
    datasets=datasets,
    overwrite=False,
    random_state=42,
    n_jobs=4,
    suffix='sfreq-48',
)

<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.channel_type is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.


To use the get_shape_from_baseconcar, InputShapeSetterEEG, BraindecodeDatasetLoaderyou need to install `braindecode`.`pip install braindecode` or Please refer to `https://braindecode.org`.


/usr/local/lib/python3.10/dist-packages/moabb/pipelines/__init__.py:26: ModuleNotFoundError: Tensorflow is not installed. You won't be able to use these MOABB pipelines if you attempt to do so.
  warn(


In [8]:
from sklearn.pipeline import Pipeline
from hoda.hoda import HODA, BTTDA, InfoBTTDA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from hoda.classification import Vectorize, ToeplitzLDAWrapper, SelectF, Tensor
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from mne.decoding import Scaler
from sklearn.linear_model import LogisticRegression
from hoda.tensorize import HankelTensor

pipelines = dict()


hoda_params = dict(
    max_iter=128,
    tol=1e-12,
    init ='random',
    random_state=42,
    shrinkage='lw',
    solver='lanczos',
    taper=False,
    forward=False,
    ortho=False,
    obj='rt',
    toeplitz=None
)

pipelines['HODA'] = Pipeline([
    ('gpu', Tensor()),
    ('clf', GridSearchCV(
        Pipeline([
            ('hoda', HODA(**hoda_params,    delta=None,)),
            ('vec', Vectorize()),
            ('zscore', StandardScaler()),
            ('lda', LDA(shrinkage='auto', solver='lsqr')),
        ]),
        param_grid=dict(hoda__rank=[1,2,4,8,16]),
        n_jobs=4
    )),
])
"""
pipelines['HODA_hankel'] = Pipeline([
    ('hankel', HankelTensor()),
    ('gpu', Tensor()),
    ('clf', GridSearchCV(
        Pipeline([
            ('hoda', HODA(**hoda_params,    delta=None,)),
            ('vec', Vectorize()),
            ('zscore', StandardScaler()),
            ('lda', LDA(shrinkage='auto', solver='lsqr')),
        ]),
        param_grid=dict(hoda__rank=[1,2,4,8,16]),
        n_jobs=1
    )),
])
"""
pipelines['tLDA'] = ToeplitzLDAWrapper()
"""
pipelines['BTTDA_bic_8_truncate'] = Pipeline([
    ('bttda', InfoBTTDA(ranks=[None]*8,hoda_params=hoda_params,info_crit='bic', truncate=True)),
    ('vec', Vectorize()),
    ('zscore', StandardScaler()),
    ('lda', LDA(shrinkage='auto', solver='lsqr')),
])
pipelines['BTTDA_bic_16_truncate'] = Pipeline([
    ('bttda', InfoBTTDA(ranks=[None]*16,hoda_params=hoda_params,info_crit='bic', truncate=True)),
    ('vec', Vectorize()),
    ('zscore', StandardScaler()),
    ('lda', LDA(shrinkage='auto', solver='lsqr')),
])
"""

"\npipelines['BTTDA_bic_8_truncate'] = Pipeline([\n    ('bttda', InfoBTTDA(ranks=[None]*8,hoda_params=hoda_params,info_crit='bic', truncate=True)),\n    ('vec', Vectorize()),\n    ('zscore', StandardScaler()),\n    ('lda', LDA(shrinkage='auto', solver='lsqr')),\n])\npipelines['BTTDA_bic_16_truncate'] = Pipeline([\n    ('bttda', InfoBTTDA(ranks=[None]*16,hoda_params=hoda_params,info_crit='bic', truncate=True)),\n    ('vec', Vectorize()),\n    ('zscore', StandardScaler()),\n    ('lda', LDA(shrinkage='auto', solver='lsqr')),\n])\n"

In [9]:
#import warnings
#warnings.filterwarnings("ignore")

results = evaluation.process(pipelines)

BNCI2014-008-WithinSession:   0%|                                                               | 0/8 [00:00<?, ?it/s]

No hdf5_path provided, models will not be saved.


/usr/local/lib/python3.10/dist-packages/sklearn/preprocessing/_function_transformer.py:394: FutureWarning: The current default of copy=False will change to copy=True in 1.7. Set the value of copy explicitly to avoid this warning
  return func(X, **(kw_args if kw_args else {}))
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_types is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_channels_regexp is deprecated will be removed in 1.6, use documented public API instead. If no appropriate public API exists, please open an issue on GitHub.
<frozen importlib._bootstrap>:241: FutureWarning: mne.io.pick.pick_

In [ ]:
results

In [ ]:
results.groupby(['dataset','session', 'pipeline']).aggregate('mean') 

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

order = results.groupby('pipeline')
order = order.score.aggregate('mean')
order = order.sort_values()


sns.catplot(data=results , x='session', y='score', col='dataset',hue='pipeline', col_wrap=3,kind='bar')
plt.show()

In [ ]:
from moabb.analysis.meta_analysis import compute_dataset_statistics, find_significant_differences
from moabb.analysis.plotting import summary_plot
import matplotlib.pyplot as plt

stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)
plt.show()

In [ ]:
from moabb.analysis.plotting import meta_analysis_plot, paired_plot
_ = meta_analysis_plot(stats,'HODA',  'BTTDA_bic_8_truncate')
plt.show()
_  = paired_plot(results, 'HODA', 'BTTDA_bic_8_truncate')
plt.show()
